# Beam search as regularized MAP decoding

Demo of Meister, Vieira & Cotterell, *If Beam Search is the Answer, What was the Question?*
(citekey `meisterIfBeamSearch2021`, [arXiv:2010.02650](https://arxiv.org/abs/2010.02650)).
Notes: `../llm-inference-notes/notes/papers/meisterIfBeamSearch2021.md`.

**Puzzle.** Decoding a neural text generator by *exact* search for the most probable output (MAP,
$\mathbf y^\star=\arg\max_{\mathbf y}\log p(\mathbf y\mid\mathbf x)$) often gives poor text, in the worst case the
empty string. Beam search, which searches badly, usually gives better text, and a *wider* beam (a better search) can
make things worse.

**Answer of the paper.** Beam search is the exact solution of a *different* objective,

$$\mathbf y^\star=\arg\max_{\mathbf y}\ \log p(\mathbf y\mid\mathbf x)-\lambda\,R(\mathbf y),
\qquad R_\text{greedy}(\mathbf y)=\sum_{t}\Big(u_t(y_t)-\min_{y'}u_t(y')\Big)^2,$$

where $u_t(y)=-\log p(y\mid\mathbf x,\mathbf y_{<t})$ is the **surprisal** at step $t$. For $\lambda\to\infty$ this is
greedy decoding, for $\lambda=0$ it is MAP. The regularizer asks every step to be close to the least surprising
choice, which spreads information evenly over the sentence: the **uniform information density** (UID) hypothesis from
cognitive science. Other UID regularizers (variance, local consistency, max, squared surprisal) are defined in the
paper and implemented below.

**What this notebook does.** It uses the paper's own setting, German→English translation with a small MarianMT model
(`Helsinki-NLP/opus-mt-de-en`), and implements everything from scratch: beam search, an *exact* best-first (Dijkstra)
search for the (regularized) MAP objective, and the regularizers.

1. Beam width versus MAP: wider beams approach the exact MAP output, which can be short or empty.
2. Regularized MAP: exact search with a UID regularizer, compared with beam search.
3. Surprisal profiles: how evenly each output spreads information.

**Honest caveat.** The MAP pathology is not universal. On clean, in-domain sentences every method returns the same
translation, and the small Qwen2.5 language models I tried first showed no degradation at all. It shows up on
unusual inputs, such as the word salad and the truncated sentence among the examples below.

## Setup

In [ ]:
import heapq
import html
import itertools
from functools import cache
from types import SimpleNamespace

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import clear_output, display

%matplotlib inline

In [ ]:
MODEL_ID = "Helsinki-NLP/opus-mt-de-en"
MAX_NEW_TOKENS = (
    60  # length cap: a hypothesis that reaches it is forced to end with EOS
)
NODE_BUDGET = (
    2000  # max decoder calls for one exact search; the search gives up beyond it
)
POOL_K = 100  # beam width whose finished hypotheses are rescored for the non-additive regularizers

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"

In [ ]:
from transformers import MarianMTModel, MarianTokenizer

tok = MarianTokenizer.from_pretrained(MODEL_ID)
model = MarianMTModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
EOS, START, PAD = (
    tok.eos_token_id,
    model.config.decoder_start_token_id,
    model.config.pad_token_id,
)

## Model access

`Source` encodes one German sentence once and then answers "what is the log-distribution over the next English
token, given this prefix?" for a batch of equal-length prefixes. There is no KV cache: sentences are short, and only
the last position goes through the output layer, which is what keeps wide beams fast.

`trace` runs one teacher-forced pass over a finished hypothesis and returns its surprisals $u_t$ (EOS included) and
the per-step minimum surprisal $\min_{y'}u_t(y')$ that the greedy regularizer needs.

In [ ]:
def detok(tokens):
    return tok.decode(tokens, skip_special_tokens=True)


class Source:
    def __init__(self, text):
        ids = tok(text, return_tensors="pt").input_ids.to(DEVICE)
        with torch.no_grad():
            self.enc = model.get_encoder()(input_ids=ids).last_hidden_state

    def _logits(self, dec_ids, last_only):
        enc = self.enc.expand(dec_ids.shape[0], -1, -1)
        h = model.model.decoder(
            input_ids=dec_ids, encoder_hidden_states=enc
        ).last_hidden_state
        if last_only:
            h = h[:, -1:]
        logits = (model.lm_head(h) + model.final_logits_bias).float()
        logits[..., PAD] = float("-inf")  # PAD doubles as the decoder start token
        return logits

    @torch.no_grad()
    def next_logprobs(self, prefixes):
        # prefixes: list of equal-length token lists -> (batch, vocab) log-probabilities on the CPU
        dec_ids = torch.tensor([[START] + p for p in prefixes], device=DEVICE)
        return torch.log_softmax(self._logits(dec_ids, True)[:, 0], -1).cpu()

    @torch.no_grad()
    def trace(self, tokens):
        # surprisals u_t of the hypothesis (EOS appended) and min_y' u_t(y') at every step
        full = tokens + [EOS]
        dec_ids = torch.tensor([[START] + full[:-1]], device=DEVICE)
        lp = torch.log_softmax(self._logits(dec_ids, False)[0], -1).cpu()
        u = -lp[torch.arange(len(full)), torch.tensor(full)]
        return u.numpy(), (-lp.max(-1).values).numpy()


@cache
def get_source(text):
    return Source(text)

## The regularizers

Each takes the surprisals `u` of a hypothesis (and `m`, the per-step minimum surprisal) and returns $R(\mathbf y)$.
Lower objective is better: $\text{cost}(\mathbf y)=-\log p(\mathbf y)+\lambda R(\mathbf y)$.

| name | $R(\mathbf y)$ | additive over steps? |
|---|---|---|
| `greedy` | $\sum_t (u_t-\min_{y'}u_t(y'))^2$ | yes |
| `squared` | $\sum_t u_t^2$ | yes |
| `variance` | $\frac1{\lvert y\rvert}\sum_t (u_t-\bar u)^2$ | no |
| `local` | $\frac1{\lvert y\rvert}\sum_t (u_t-u_{t-1})^2$ with $u_0=0$ | no |
| `max` | $\max_t u_t$ | no |

*Additive* means the cost of a prefix only grows as it is extended and decomposes into a per-token cost. Then
Dijkstra's algorithm is exact and beam search can use the regularized score directly. For the other three I
**rescore** the finished hypotheses of a wide beam instead. That is not exact search; the paper stays exact with a
bound-based stopping rule, which this notebook does not implement.

In [ ]:
REGULARIZERS = {
    "greedy": lambda u, m: float(((u - m) ** 2).sum()),
    "squared": lambda u, m: float((u**2).sum()),
    "variance": lambda u, m: float(u.var()),
    "local": lambda u, m: float(((u - np.concatenate([[0.0], u[:-1]])) ** 2).mean()),
    "max": lambda u, m: float(u.max()),
}
ADDITIVE = ("greedy", "squared")


def objective(u, m, reg, lam):
    # cost = -log p(y) + lam * R(y), lower is better
    return float(u.sum()) + (lam * REGULARIZERS[reg](u, m) if reg else 0.0)


def token_cost(u, reg, lam):
    # per-token cost for the additive regularizers; u is the surprisal over the whole vocabulary
    if reg is None or lam == 0:
        return u
    if reg == "greedy":
        return u + lam * (u - u.min()) ** 2
    if reg == "squared":
        return u + lam * u**2
    raise ValueError(f"{reg} is not additive")

## Beam search

Exactly the recursion of the paper (Eq. 4-6): keep a set of $k$ hypotheses, extend each unfinished one by every
token, and keep the $k$ best candidates. Finished hypotheses (ending in EOS) stay in the set and compete, so the
search stops once all $k$ are finished. With `reg=None` it maximizes $\log p$; with an additive regularizer it uses
the regularized per-token cost. $k=1$ is greedy decoding. There is no length normalization.

In [ ]:
@cache
def beam_search(text, k, reg=None, lam=0.0):
    # returns the final set of finished hypotheses as [(tokens, cost)], best first
    src = get_source(text)
    active, done = [([], 0.0)], []
    for step in range(MAX_NEW_TOKENS + 1):
        cands = [(c, seq, True) for seq, c in done]
        lps = src.next_logprobs([seq for seq, _ in active])
        for (seq, c), lp in zip(active, lps):
            cost = token_cost(-lp, reg, lam)
            if step == MAX_NEW_TOKENS:  # length cap: only EOS is allowed
                cands.append((c + cost[EOS].item(), seq, True))
                continue
            top = cost.topk(k, largest=False)
            for v, t in zip(top.values.tolist(), top.indices.tolist()):
                if v < float("inf"):
                    cands.append((c + v, seq + [t] if t != EOS else seq, t == EOS))
        cands.sort(key=lambda x: x[0])
        top_k = cands[:k]
        done = [(seq, c) for c, seq, fin in top_k if fin]
        active = [(seq, c) for c, seq, fin in top_k if not fin]
        if not active:
            break
    return sorted(done, key=lambda h: h[1])

## Exact search

Best-first (Dijkstra) search over prefixes, as in the paper. A node is a prefix, the edge to a child costs
`token_cost`, a hypothesis is complete at EOS, and the first complete hypothesis popped from the queue is optimal
because costs never decrease along a path (this needs the additive regularizers).

Children are expanded **lazily**: when a prefix is first popped, one decoder call ranks all of its children by
cost, but only the cheapest child is queued, together with a pointer to the next sibling. Decoder calls therefore
happen only for prefixes whose cost is below the optimum. A node budget stops the search on hard inputs and says so
instead of returning a wrong answer.

In [ ]:
class SearchBudgetExceeded(Exception):
    pass


def _exact_search(text, reg, lam):
    src = get_source(text)
    tie = itertools.count()
    heap = [(0.0, next(tie), "node", ([],))]
    calls = 0
    while heap:
        cost, _, kind, payload = heapq.heappop(heap)
        if kind == "final":
            return payload[0], calls
        if kind == "node":
            (seq,) = payload
            calls += 1
            if calls > NODE_BUDGET:
                raise SearchBudgetExceeded(f"more than {NODE_BUDGET} decoder calls")
            lp = src.next_logprobs([seq])[0]
            step_cost = token_cost(-lp, reg, lam)
            if len(seq) == MAX_NEW_TOKENS:  # length cap: only EOS is allowed
                heapq.heappush(
                    heap, (cost + step_cost[EOS].item(), next(tie), "final", (seq,))
                )
                continue
            order = torch.argsort(step_cost)
            first = step_cost[order[0]].item()
            heapq.heappush(
                heap,
                (cost + first, next(tie), "child", (seq, cost, step_cost, order, 0)),
            )
        else:  # "child": child j of a node; queue sibling j + 1, then the child itself
            seq, base, step_cost, order, j = payload
            if j + 1 < len(order) and step_cost[order[j + 1]] < float("inf"):
                nxt = base + step_cost[order[j + 1]].item()
                heapq.heappush(
                    heap,
                    (nxt, next(tie), "child", (seq, base, step_cost, order, j + 1)),
                )
            t = order[j].item()
            if t == EOS:
                heapq.heappush(heap, (cost, next(tie), "final", (seq,)))
            else:
                heapq.heappush(heap, (cost, next(tie), "node", (seq + [t],)))
    raise RuntimeError("search space exhausted")


@cache
def exact_search(text, reg=None, lam=0.0):
    # tokens of the optimum of  -log p(y) + lam * R(y)  (lam = 0 is plain MAP), or None if over budget
    if reg not in (None, *ADDITIVE):
        raise ValueError(f"{reg} is not additive; use rescored_pool")
    try:
        return _exact_search(text, reg, lam)[0]
    except SearchBudgetExceeded:
        return None


def rescored_pool(text, reg, lam):
    # best of the POOL_K finished beam hypotheses under  -log p(y) + lam * R(y)  (not exact search)
    src = get_source(text)
    pool = [tokens for tokens, _ in beam_search(text, POOL_K)]
    return min(pool, key=lambda t: objective(*src.trace(t), reg, lam))

## Reporting helpers

In [ ]:
PALETTE = {
    "greedy": "#e69f00",
    "beam": "#1b9e77",
    "exact MAP": "#7b3294",
    "exact reg": "#d95f02",
    "beam reg": "#0072b2",
}


def describe(text, name, tokens, reg=None, lam=0.0, kind=None):
    u, m = get_source(text).trace(tokens)
    return SimpleNamespace(
        name=name,
        kind=kind or name,
        text=detok(tokens),
        n=len(tokens),
        logp=-float(u.sum()),
        std=float(u.std()),
        cost=objective(u, m, reg, lam),
        u=u,
    )


def table_html(rows, with_cost=False):
    head = ["method", "translation", "log p", "tokens", "std(u)"] + (
        ["objective"] if with_cost else []
    )
    out = ["<table style='border-collapse:collapse'><tr>"]
    out += [f"<th style='text-align:left;padding:2px 10px'>{h}</th>" for h in head]
    out.append("</tr>")
    for r in rows:
        shown = html.escape(r.text) if r.text else "<i>(empty string)</i>"
        cells = [r.name, shown, f"{r.logp:.2f}", r.n, f"{r.std:.2f}"]
        if with_cost:
            cells.append(f"{r.cost:.2f}")
        out.append("<tr>")
        out += [f"<td style='padding:2px 10px'>{c}</td>" for c in cells]
        out.append("</tr>")
    out.append("</table>")
    return "".join(out)

## Examples

Clean, in-domain sentences are translated identically by every method. The last three are the interesting ones:
a word salad of German function words, a center-embedded sentence, and a truncated sentence.

In [ ]:
EXAMPLES = {
    "clean: coffee": "Ich möchte eine Tasse Kaffee, bitte.",
    "clean: treaty": "Der Vertrag tritt am Tag nach seiner Veröffentlichung im Amtsblatt der Europäischen Union in Kraft.",
    "clean: reported speech": "Er sagte, dass er das Buch, das ich ihm geliehen hatte, leider nicht gelesen habe.",
    "hard: function words": "und der die das ein eine einer eines einem einen",
    "hard: center embedding": "Der Mann, der den Hund, der die Katze, die die Maus, die den Käse gefressen hatte, gejagt hatte, gebissen hatte, sah, lachte.",
    "hard: truncated": "Ich möchte",
}

## Part 1: wider beams approach exact MAP

For one sentence: beam search with growing width $k$ on the plain MAP objective, and the exact MAP output at the end
(a beam of unbounded width). Log-probability can only go up as the search improves, so whatever changes in the text
changes *because the search is better*, not worse.

In [ ]:
def beam_width_report(text, ks=(1, 2, 5, 10, 20, 50, 100)):
    rows = [
        describe(text, f"beam k={k}", beam_search(text, k)[0][0], kind="beam")
        for k in ks
    ]
    exact = exact_search(text)
    if exact is not None:
        rows.append(describe(text, "exact MAP", exact, kind="exact MAP"))
    display(widgets.HTML(f"<b>{html.escape(text)}</b>" + table_html(rows)))
    if exact is None:
        print(f"exact MAP: gave up after {NODE_BUDGET} decoder calls (long sentence)")

    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    x = list(range(len(ks)))
    labels = [str(k) for k in ks]
    for ax, key, label in zip(
        axes, ("logp", "n"), ("log p of the output", "length (tokens)")
    ):
        ax.plot(
            x, [getattr(r, key) for r in rows[: len(ks)]], "o-", color=PALETTE["beam"]
        )
        if exact is not None:
            ax.axhline(
                getattr(rows[-1], key),
                color=PALETTE["exact MAP"],
                ls="--",
                label="exact MAP",
            )
        ax.set_xticks(x, labels)
        ax.set_xlabel("beam width k")
        ax.set_ylabel(label)
    axes[0].legend()
    fig.tight_layout()
    plt.show()


for name in ("hard: function words", "hard: truncated", "clean: reported speech"):
    beam_width_report(EXAMPLES[name])

## Part 2 and 3: regularized MAP, and how evenly information is spread

Pick a sentence (or type your own), a regularizer and a strength $\lambda$, then press **Run**.

* **greedy**: $k=1$.
* **beam k**: plain beam search of the chosen width.
* **exact MAP**: the $\lambda=0$ optimum.
* **exact reg**: the optimum of $\log p-\lambda R$. For `variance`, `local` and `max` this is instead the best of the
  $k=100$ finished beam hypotheses under the regularized objective (not exact).
* **beam reg**: beam search of the chosen width, scored with the regularized objective (additive regularizers only).

The table reports the log-probability, the length, the standard deviation of the surprisals (the paper's measure of
UID; lower is more uniform) and the regularized objective. The plot shows the surprisal $u_t$ at each output
position. Things to look for: the exact MAP output pays for a high log-probability with a few very surprising
steps or a very short output, while larger $\lambda$ pulls the exact solution towards the beam search output.

In [ ]:
def run_methods(text, k, reg, lam):
    rows, notes = [], []
    rows.append(
        describe(text, "greedy", beam_search(text, 1)[0][0], reg, lam, "greedy")
    )
    rows.append(
        describe(text, f"beam k={k}", beam_search(text, k)[0][0], reg, lam, "beam")
    )
    exact = exact_search(text)
    if exact is None:
        notes.append(f"exact MAP: gave up after {NODE_BUDGET} decoder calls")
    else:
        rows.append(describe(text, "exact MAP", exact, reg, lam, "exact MAP"))
    if lam > 0 and reg in ADDITIVE:
        tokens = exact_search(text, reg, lam)
        if tokens is None:
            notes.append(f"exact reg: gave up after {NODE_BUDGET} decoder calls")
        else:
            rows.append(describe(text, f"exact {reg}", tokens, reg, lam, "exact reg"))
        top = beam_search(text, k, reg, lam)[0][0]
        rows.append(describe(text, f"beam k={k} + {reg}", top, reg, lam, "beam reg"))
    elif lam > 0:
        tokens = rescored_pool(text, reg, lam)
        rows.append(
            describe(text, f"rescored pool ({reg})", tokens, reg, lam, "exact reg")
        )
    return rows, notes


def render(text, k, reg, lam):
    rows, notes = run_methods(text, k, reg, lam)
    display(widgets.HTML(table_html(rows, with_cost=True)))
    for note in notes:
        print(note)
    fig, ax = plt.subplots(figsize=(9, 3.4))
    for r in rows:
        x = np.arange(1, len(r.u) + 1)
        ax.plot(
            x,
            r.u,
            "o-",
            ms=3,
            color=PALETTE[r.kind],
            label=f"{r.name} (std {r.std:.2f})",
        )
    ax.set_xlabel("output position (last point is EOS)")
    ax.set_ylabel("surprisal u_t (nats)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

In [ ]:
example_w = widgets.Dropdown(
    options=list(EXAMPLES), value="hard: function words", description="example"
)
custom_w = widgets.Text(placeholder="or type German text", description="custom")
k_w = widgets.Dropdown(
    options=[1, 2, 5, 10, 20, 50, 100], value=5, description="beam k"
)
reg_w = widgets.Dropdown(
    options=list(REGULARIZERS), value="greedy", description="regularizer"
)
lam_w = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=5.0,
    step=0.05,
    description="lambda",
    continuous_update=False,
)
run_w = widgets.Button(description="Run", button_style="primary")
out_w = widgets.Output()


def on_run(_):
    with out_w:
        clear_output(wait=True)
        text = custom_w.value.strip() or EXAMPLES[example_w.value]
        print(text)
        render(text, k_w.value, reg_w.value, lam_w.value)


run_w.on_click(on_run)
display(widgets.VBox([example_w, custom_w, k_w, reg_w, lam_w, run_w, out_w]))

### Exact search with growing regularization

The analogue of the paper's Fig. 1: the exact optimum of $\log p-\lambda R_\text{greedy}$ for a sweep of $\lambda$.
At $\lambda=0$ it is the MAP output; as $\lambda$ grows it moves towards greedy decoding, and the surprisals become
more uniform (lower std). The paper pairs this with BLEU on a test set; here there are no reference translations, so
read the translations in the table instead.

In [ ]:
def lambda_sweep(text, reg="greedy", lams=(0, 0.05, 0.1, 0.2, 0.5, 1, 2, 5)):
    rows = []
    for lam in lams:
        tokens = exact_search(text, reg if lam else None, float(lam))
        if tokens is None:
            print(f"lambda={lam}: gave up after {NODE_BUDGET} decoder calls")
            continue
        r = describe(text, f"lambda={lam}", tokens, reg, float(lam))
        r.lam = lam
        rows.append(r)
    display(widgets.HTML(f"<b>{html.escape(text)}</b>" + table_html(rows)))
    fig, axes = plt.subplots(1, 2, figsize=(9, 3))
    for ax, key, label in zip(
        axes, ("std", "n"), ("std of surprisals", "length (tokens)")
    ):
        ax.plot(
            [r.lam for r in rows],
            [getattr(r, key) for r in rows],
            "o-",
            color=PALETTE["exact reg"],
        )
        ax.set_xlabel(f"lambda ({reg} regularizer)")
        ax.set_ylabel(label)
        ax.set_xscale("symlog", linthresh=0.05)
    fig.tight_layout()
    plt.show()


for name in ("hard: function words", "hard: truncated", "clean: reported speech"):
    lambda_sweep(EXAMPLES[name])

## Things to try

* **Function words, beam width.** Compare $k=5$ with $k=100$: the wide beam finds the empty string that exact MAP
  also returns. Nothing is wrong with the search; the objective prefers an empty output over any translation of this
  word salad. Then raise $\lambda$ with the `greedy` regularizer and watch the empty output disappear. With
  `variance` it does not: an output with a single surprisal (the lone EOS) has zero variance, so a pure UID measure
  cannot rule out the empty string, which is one reason the paper's best regularizers also penalize high surprisal.
* **Truncated sentence.** MAP prefers the *shorter* "I want" over "I want to", because the extra token costs
  probability. With the `greedy` regularizer at $\lambda=0.5$ the exact optimum is "I want to".
* **Clean sentences.** Everything coincides. The regularizer matters when the model is poorly calibrated, not when
  it is confident.
* **Center embedding.** The wide beam gives a shorter output that drops clauses; exact MAP exceeds the node budget
  here (the message under the table). Raising `NODE_BUDGET` makes it succeed but slow.
* **Std of surprisals.** The empty output has std 0 (one surprisal), so a low std alone does not mean a good
  translation; read it together with the text and the log-probability.
* **Regularizers.** Compare `greedy` and `squared` (exact) with `variance`, `local` and `max` (rescored pool): the
  latter cannot go below what the 100-wide beam contains.
* **Surprisal profile.** The paper's claim in one picture: beam search and well-tuned regularized outputs have a
  smaller std of surprisals than the MAP output.